# MHT-CET College Predictor
### Engineering CAP Round 1 — cutoff analysis & seat-allocation predictor

**Goal.** Given a student's percentile, social category, gender, and home-university status,
predict the colleges and branches they can realistically expect a seat in.

**Data.** Official MHT-CET CAP Round 1 cutoff lists. Each record is one
*(college, branch, seat-type, reservation category)* combination with the **closing percentile**
(percentile of the last candidate admitted) and the **closing merit number**.

**Scope & honesty note.**
- A single year lets us build a predictor that assumes *next year ≈ this year* (a standard, reasonable baseline).
- With **2022 + 2023 + 2024** loaded, we additionally model how cutoffs *move* year-over-year and project an
  expected 2025 cutoff per option. That projection is the genuine forecasting component.
- This notebook auto-detects which years are present and adapts.


## 1. Setup

In [ ]:
import glob, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.grid'] = True

## 2. Load the data

Drop the cutoff CSV files (one per year, named like `mhtcet_2024_cap1_cutoffs.csv`) next to this
notebook. The loader picks up every year it finds and stacks them into one frame with a `year` column.

> The PDF→CSV parser used to produce these files is included at the **end** of the notebook (Appendix)
> for full transparency; you don't need to run it here.

In [ ]:
CSV_GLOB = 'mhtcet_2022_cap1_cutoffs.csv'
CSV_GLOB = 'mhtcet_2023_cap1_cutoffs.csv'
CSV_GLOB = 'mhtcet_2024_cap1_cutoffs.csv'


def load_all_years(pattern=CSV_GLOB):
    frames = []
    for path in sorted(glob.glob(pattern)):
        m = re.search(r'(20\d{2})', os.path.basename(path))
        if not m:
            continue
        d = pd.read_csv(path)
        d['year'] = int(m.group(1))
        frames.append(d)
    if not frames:
        raise FileNotFoundError(f"No files matching {pattern} found in this folder.")
    df = pd.concat(frames, ignore_index=True)
    # tidy types
    df['closing_percentile'] = pd.to_numeric(df['closing_percentile'], errors='coerce')
    df['closing_merit_no']   = pd.to_numeric(df['closing_merit_no'], errors='coerce')
    return df

df = load_all_years()
years = sorted(df['year'].unique())
print('Years loaded:', years)
print('Rows:', len(df), '| Colleges:', df.college_code.nunique(),
      '| Branches:', df.groupby(['college_code','branch_code']).ngroups,
      '| Categories:', df.category.nunique())
df.head()

In [ ]:
REGION_BY_PREFIX = {
    '01': 'Amravati', '02': 'Chhatrapati Sambhajinagar', '03': 'Mumbai',
    '04': 'Nagpur',   '05': 'Nashik',                    '06': 'Pune',
}
CITY_REGION = {  # fallback for special university codes (COEP, etc.)
    'navi mumbai': 'Mumbai', 'mumbai': 'Mumbai', 'pune': 'Pune', 'coep': 'Pune',
    'nagpur': 'Nagpur', 'nashik': 'Nashik', 'amravati': 'Amravati',
    'sambhajinagar': 'Chhatrapati Sambhajinagar', 'aurangabad': 'Chhatrapati Sambhajinagar',
}

def add_region(frame):
    f = frame.copy()
    f['region'] = f['college_code'].astype(str).str.zfill(5).str[:2].map(REGION_BY_PREFIX)
    mask = f['region'].isna()
    def by_city(name):
        n = str(name).lower()
        for k, v in CITY_REGION.items():
            if k in n:
                return v
        return 'Other'
    f.loc[mask, 'region'] = f.loc[mask, 'college_name'].apply(by_city)
    return f

df = add_region(df)
expected = add_region(expected)
print('Regions:', sorted(expected.region.unique()))

## 3. Understanding the reservation-category codes

Each `category` code packs three pieces of information:

| Part | Position | Meaning |
|------|----------|---------|
| Gender | first letter | **G** = General/gender-neutral, **L** = Ladies (female-only seats) |
| Social category | middle | OPEN, SC, ST, OBC, EWS, VJ (=VJA/DT), NT1, NT2, NT3, SEBC, … |
| Seat scope | last letter | **S** = State level, **H** = Home university, **O** = Other-than-home |

Examples: `GOPENS` = General-Open, State-level · `LOBCH` = Ladies-OBC, Home-university ·
`GSCO` = General-SC, Other-than-home.

The helper below turns a *student profile* into the **set of category codes that student is allowed to
compete for** — including the rule that reserved-category candidates also compete for OPEN seats, and
that female candidates are eligible for both Ladies and gender-neutral seats.

In [ ]:
SOCIAL_TO_CODE = {
    'OPEN':'OPEN','SC':'SC','ST':'ST','OBC':'OBC','EWS':'EWS',
    'VJ':'VJ','VJA':'VJ','DT':'VJ','NT1':'NT1','NT2':'NT2','NT3':'NT3','SEBC':'SEBC',
}

def eligible_categories(social='OPEN', gender='male', home_university=True, df=df):
    """Return the set of cutoff category codes a student may compete for.

    Assumptions (documented & simplifiable):
      * reserved candidates also compete for OPEN seats
      * female candidates compete for BOTH Ladies (L*) and gender-neutral (G*) seats
      * home-university students -> State (S) + Home (H) seats; others -> State (S) + Other (O)
    """
    social = social.upper()
    mids = {SOCIAL_TO_CODE.get(social, 'OPEN')}
    mids.add('OPEN')                      # everyone is eligible for OPEN
    if social == 'EWS':
        mids = {'EWS', 'OPEN'}
    genders = ['G', 'L'] if gender.lower().startswith('f') else ['G']
    scopes = ['S', 'H'] if home_university else ['S', 'O']
    wanted = {f'{g}{m}{s}' for g in genders for m in mids for s in scopes}
    # keep only codes that actually occur in the data
    return sorted(wanted & set(df['category'].unique()))

eligible_categories('OBC', 'female', home_university=True)

## 4. Exploratory analysis

In [ ]:
latest = max(years)
cur = df[df.year == latest]

# 4a. Distribution of OPEN-category closing percentiles (how competitive seats are)
opens = cur[cur.category.isin(['GOPENS','GOPENH','GOPENO'])]
plt.hist(opens.closing_percentile.dropna(), bins=40, color='#4C72B0', edgecolor='white')
plt.title(f'Closing percentile distribution — OPEN seats, {latest}')
plt.xlabel('Closing percentile'); plt.ylabel('# of college-branch seats')
plt.show()

In [ ]:
# 4b. Most competitive branches (highest median OPEN closing percentile)
top_branch = (opens.groupby('branch_name')['closing_percentile']
                   .median().sort_values(ascending=False).head(12))
top_branch.iloc[::-1].plot(kind='barh', color='#55A868')
plt.title(f'Most competitive branches by median OPEN cutoff ({latest})')
plt.xlabel('Median closing percentile'); plt.tight_layout(); plt.show()

In [ ]:
# 4c. Most competitive colleges (top by median OPEN cutoff, min 3 branches)
g = opens.groupby('college_name')['closing_percentile']
top_college = g.median()[g.count() >= 3].sort_values(ascending=False).head(15)
top_college.iloc[::-1].plot(kind='barh', color='#C44E52')
plt.title(f'Most competitive colleges by median OPEN cutoff ({latest})')
plt.xlabel('Median closing percentile'); plt.tight_layout(); plt.show()

## 5. Year-over-year cutoff trend & 2025 projection

*This section activates automatically only when more than one year is loaded.*

For every *(college, branch, category)* we look at how the closing percentile changed across years and
estimate the **typical yearly drift**. The projected next-year cutoff is the latest value plus the median
drift (clipped to the valid 0–100 range). This projected cutoff is what the predictor uses when multiple
years are available — it accounts for branches that are trending hotter or cooling off.

In [ ]:
KEY = ['college_code','branch_code','category']

def build_expected_cutoff(df):
    latest = max(df.year.unique())
    if df.year.nunique() == 1:
        out = df[df.year == latest].copy()
        out['expected_cutoff'] = out['closing_percentile']
        out['trend'] = 0.0
        return out

    # pivot years to columns to compute per-key drift
    piv = df.pivot_table(index=KEY, columns='year', values='closing_percentile')
    yrs = sorted(piv.columns)
    drift = piv[yrs].diff(axis=1).median(axis=1)          # median year-over-year change
    drift = drift.clip(-8, 8)                              # cap implausible swings from sparse history
    last_val = piv[yrs].ffill(axis=1)[yrs[-1]]
    expected = (last_val + drift).clip(0, 100)

    proj = pd.DataFrame({'expected_cutoff': expected, 'trend': drift}).reset_index()
    # attach names/seat_type from the latest year
    meta = (df[df.year == latest][KEY + ['college_name','branch_name','seat_type']]
            .drop_duplicates(KEY))
    out = proj.merge(meta, on=KEY, how='left')
    out['closing_percentile'] = last_val.reindex(
        out.set_index(KEY).index).values
    return out

expected = build_expected_cutoff(df)
print(f"Expected-cutoff table built for {len(expected)} options "
      f"({'projected from trend' if df.year.nunique()>1 else 'latest year only'}).")
expected.head()

## 6. The college predictor

Given a student profile, return every eligible option whose **expected cutoff** is within reach, bucketed
into three confidence bands:

- **Safe** — expected cutoff is comfortably below the student's percentile (high chance).
- **Target** — expected cutoff is just below the percentile (likely, depends on the year).
- **Reach** — expected cutoff is slightly *above* the percentile (possible if cutoffs dip this year).

For each college-branch we keep the *easiest* eligible seat (lowest cutoff among the student's categories).

In [ ]:
def predict_colleges(percentile, social='OPEN', gender='male', home_university=True,
                     branch=None, region=None, reach=1.0, target=2.0, safe=4.0,
                     top_n=None, table=expected):
    """branch: substring of branch name. region: e.g. 'Pune', 'Mumbai', 'Nagpur',
    'Nashik', 'Amravati', 'Chhatrapati Sambhajinagar'."""
    codes = eligible_categories(social, gender, home_university)
    d = table[table.category.isin(codes) & table.expected_cutoff.notna()].copy()

    if branch:
        branches = [branch] if isinstance(branch, str) else list(branch)
        pattern = '|'.join(re.escape(b) for b in branches)
        d = d[d.branch_name.str.contains(pattern, case=False, na=False, regex=True)]
    if region:
        d = d[d.region.str.contains(region, case=False, na=False)]

    d = (d.sort_values('expected_cutoff')
           .groupby(['college_code', 'branch_code'], as_index=False).first())

    d['margin'] = percentile - d['expected_cutoff']
    d = d[d['margin'] >= -reach]

    def band(m):
        if m >= safe:   return '1. Safe'
        if m >= 0:      return '2. Target'
        return '3. Reach'
    d['chance'] = d['margin'].apply(band)

    cols = ['chance', 'college_name', 'region', 'branch_name', 'category', 'expected_cutoff', 'margin']
    if 'trend' in d.columns:
        cols.append('trend')
    out = (d[cols].sort_values('expected_cutoff', ascending=False)
                 .reset_index(drop=True))
    out['expected_cutoff'] = out['expected_cutoff'].round(3)
    out['margin'] = out['margin'].round(3)
    return out.head(top_n) if top_n else out

### Example — a 92.5-percentile OBC female, home university

In [ ]:
result = predict_colleges(percentile=92.5, social='OBC', gender='female', home_university=True)
print('Total options:', len(result))
print(result['chance'].value_counts().sort_index().to_string())
result.head(25)

### Try your own profile
Edit the values below and re-run.

In [ ]:
predict_colleges(
    percentile=80.0,
    social='SC',
    gender='male',
    branch=['Information Technology', 'Computer Engineering', 'Mechanical Engineering'],
    region='Pune',
).head(10)

## 7. ML model — learning the cutoff surface

Beyond the rule-based predictor, we train a model to **predict the closing percentile** of an option from
its features (college, branch, category, seat scope, and year). Uses:

1. **Imputation** — estimate a cutoff for college-branch-category combinations that had no allotment / are
   missing, so the predictor isn't blind to them.
2. **Forecasting** — once `year` is a feature, predicting with `year = next_year` gives a model-based
   cutoff projection to compare against the trend method in §5.

We use gradient-boosted trees and report MAE / R² on a held-out split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, r2_score

model_df = df.dropna(subset=['closing_percentile']).copy()
model_df['scope'] = model_df['category'].str[-1]                 # S / H / O
model_df['social'] = model_df['category'].str.replace(r'^[GL]', '', regex=True).str[:-1]
model_df['gender_seat'] = model_df['category'].str[0]            # G / L

features = ['college_name','branch_name','social','scope','gender_seat','year']
X, y = model_df[features], model_df['closing_percentile']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

pre = ColumnTransformer([('cat',
        OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
        features)])
reg = Pipeline([('pre', pre),
                ('gb', HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08,
                                                     random_state=42))])
reg.fit(Xtr, ytr)
pred = reg.predict(Xte)
print(f"MAE: {mean_absolute_error(yte, pred):.2f} percentile points")
print(f"R² : {r2_score(yte, pred):.3f}")

In [ ]:
# Feature signal check: predict for a few profiles
demo = pd.DataFrame([
    {'college_name': model_df.college_name.mode()[0], 'branch_name':'Computer Science and Engineering',
     'social':'OPEN','scope':'S','gender_seat':'G','year':max(years)},
    {'college_name': model_df.college_name.mode()[0], 'branch_name':'Civil Engineering',
     'social':'OPEN','scope':'S','gender_seat':'G','year':max(years)},
])
demo['predicted_cutoff'] = reg.predict(demo).clip(0, 100).round(2)
demo

## 8. Limitations & next steps

- **Eligibility is simplified.** Real CAP eligibility also depends on minority/linguistic status,
  PWD/Defence/Orphan sub-quotas, and TFW seats. Extend `eligible_categories` if you need those.
- **Cutoffs are CAP Round 1 only.** Round 2/3 and institute-level rounds shift seats further; a student
  slightly short in Round 1 often gets in later. Treat "Reach" generously.
- **Projection needs more years for confidence.** With 3 years the median-drift estimate is rough; 4–5
  years would let you fit and validate a proper per-option trend.
- **Possible extensions:** add a Streamlit front-end over `predict_colleges`; cluster colleges by cutoff
  profile; add geographic/region features; calibrate the safe/target/reach margins against actual Round-2
  movement.

## Appendix — PDF → CSV parser (for reproducibility)

This is the validated parser that converts an official cutoff PDF into the CSV format above. It needs
`pdftotext` (from poppler-utils: `apt install poppler-utils` or `brew install poppler`). You normally
don't run this — the CSVs are already provided — but it documents exactly how the data was produced.

In [ ]:
import subprocess, csv, tempfile

SEAT_TYPES = {
    "State Level",
    "Home University Seats Allotted to Home University Candidates",
    "Home University Seats Allotted to Other Than Home University Candidates",
    "Other Than Home University Seats Allotted to Home University Candidates",
    "Other Than Home University Seats Allotted to Other Than Home University Candidates",
}

def parse_cutoff_pdf(pdf_path, out_csv):
    """Convert an MHT-CET CAP cutoff PDF into the standard CSV. Returns row count."""
    txt = subprocess.run(['pdftotext', '-layout', pdf_path, '-'],
                         capture_output=True, text=True).stdout
    lines = txt.split('\n')
    branch_re  = re.compile(r'^\s*(\d{10})\s*-\s*(.+?)\s*$')
    college_re = re.compile(r'^\s*(\d{5})\s*-\s*(.+?)\s*$')
    hdr_re     = re.compile(r'^(\s*Stage\s+)(.+)$')
    data_re    = re.compile(r'^\s*(I{1,3}|IV|V)\s+\d')

    def toks(line, pat):
        return [(m.group(0), (m.start()+m.end())//2) for m in re.finditer(pat, line)]
    def nearest(centers, items):
        out = {}
        for t, c in items:
            out[min(range(len(centers)), key=lambda k: abs(centers[k]-c))] = t
        return out

    rows = []
    cc = cn = bc = bn = st = None
    i = 0
    while i < len(lines):
        ln = lines[i]; s = ln.strip()
        if (m := branch_re.match(ln)): bc, bn = m.groups(); i += 1; continue
        if (m := college_re.match(ln)) and len(m.group(1)) == 5: cc, cn = m.groups(); i += 1; continue
        if s in SEAT_TYPES: st = s; i += 1; continue
        if (hm := hdr_re.match(ln)):
            off = hm.start(2)
            cat_items = [(t, c+off) for t, c in toks(ln[off:], r'[A-Z][A-Z0-9]+')]
            cats = [t for t, _ in cat_items]; centers = [c for _, c in cat_items]
            j = i + 1
            while j < min(i+6, len(lines)) and not data_re.match(lines[j]): j += 1
            if j < min(i+6, len(lines)):
                vline = re.sub(r'^(\s*)(I{1,3}|IV|V)(\s)',
                               lambda m: m.group(1)+' '*len(m.group(2))+m.group(3), lines[j])
                vmap = nearest(centers, toks(vline, r'\d+'))
                k = j + 1
                while k < min(j+4, len(lines)) and '(' not in lines[k]: k += 1
                pmap = nearest(centers, toks(lines[k], r'\d+\.\d+')) if k < len(lines) else {}
                for idx, cat in enumerate(cats):
                    rows.append([cc, cn, bc, bn, st, cat, vmap.get(idx,''), pmap.get(idx,'')])
                i = k + 1; continue
        i += 1

    with open(out_csv, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['college_code','college_name','branch_code','branch_name',
                    'seat_type','category','closing_merit_no','closing_percentile'])
        w.writerows(rows)
    return len(rows)

# Example:
# parse_cutoff_pdf('2024ENGG_CAP1_CutOff.pdf', 'mhtcet_2024_cap1_cutoffs.csv')